# Fine-tune da Laya no domínio wiki-docs (GA4) — Colab T4

Adaptado do notebook oficial `laya_finetune_typed_decisions_2xT4_kaggle.ipynb`
([repo laya](https://github.com/NandhaKishorM/laya)) para uma T4 única e o
dataset do repositório wiki-docs (`laya-treino/`).

**Como usar**
1. Runtime → Change runtime type → **T4 GPU**
2. Arraste pro painel lateral (root): `distill_treino.jsonl`,
   `distill_er_treino.jsonl`, `er_volume.jsonl` (Exp B v2 — ER balanceado),
   `classificacao_heldout.jsonl`, `fila_610.jsonl`,
   `baseline_typed-decisions.json`
3. Rode as células em ordem — o preprocess varre o `/content/`, acha os
   arquivos e monta o treino (soft labels do Jev detectados sozinhos)
4. No fim, a célula de gates compara com a baseline — decisão por
   SUCESSO.md, não por entusiasmo
5. (Opcional) push pro Hub — o checkpoint vira um subfolder e a adoção
   no pipeline é trocar `subfolder="typed-decisions"` pelo novo.

## 0. GPU

In [ ]:
!nvidia-smi

## 1. Install

In [ ]:
!pip install -q -U "laya>=0.3.20" "transformers>=4.48" "datasets>=3.0" safetensors huggingface_hub

## 1. Dados

Formato `laya-evals` (`{state, questions, expected, tags}`). Dois modos,
mesmo preprocess:

- **Exp A** (one-hot): `expected` vira alvo duro — o fine-tune clássico
  com os rótulos que temos.
- **Exp B** (destilação, recomendado): arquivos `distill_*` trazem um
  campo `soft` com as probabilidades do Jev (professor: 0.935 no
  `ga4_current` do held-out, aprovado). O alvo vira a distribuição do
  professor — que é o que o treino RLCD (`proper_reward`) espera
  nativamente. Gold humano continua alvo duro; `doc_kind` nunca entra
  em soft (professor é pior que o aluno nesse eixo, medido).

O held-out **não entra no treino** em nenhum dos dois: é o gate.

In [ ]:
import os, json, random, torch
from transformers import AutoTokenizer
from huggingface_hub import snapshot_download
from laya.agent import _fix_tokenizer_config
from laya.common import build_sequence, render_options, QTYPES

MODEL_ID = "convaiinnovations/laya"
model_dir = snapshot_download(MODEL_ID, allow_patterns=["typed-decisions/*",
                                                        "tokenizer/*",
                                                        "rl_agent_config.json"])
# o config fica na raiz do snapshot; o checkpoint em typed-decisions/
td_dir = os.path.join(model_dir, "typed-decisions")
_fix_tokenizer_config(model_dir)

tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
with open(os.path.join(model_dir, "rl_agent_config.json")) as f:
    cfg = json.load(f)

def carregar_jsonl(caminho):
    with open(caminho, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

def gold_probs(q, esperado, soft=None):
    """Distribuicao alvo da pergunta. Exp B (destilacao): quando o
    registro traz 'soft' (probabilidades do Jev, professor aprovado no
    held-out), o alvo e a distribuicao do professor — e o que o loop
    RLCD espera nativamente (proper_reward casa distribuicoes). Exp A
    e gold humano caem no one-hot. doc_kind nunca vem em soft: o
    professor e pior que o aluno nesse eixo, medido."""
    t = q["type"]
    if soft and q.get("_qid") in soft:
        s = soft[q["_qid"]]
        if t == "choice":
            keys = list(q.get("criteria", {}).keys())
            alvo = [float(s.get(k, 0.0)) for k in keys]
        else:
            keys = ["false", "true"]
            alvo = [float(s.get("false", 0.5)), float(s.get("true", 0.5))]
        total = sum(alvo)
        if total > 0:
            return keys, [v / total for v in alvo]
    if t == "choice":
        keys = list(q.get("criteria", {}).keys())
        alvo = [0.0] * len(keys)
        if esperado in keys:
            alvo[keys.index(esperado)] = 1.0
        else:
            alvo[0] = 1.0  # rotulo fora das opcoes: nao deve acontecer
        return keys, alvo
    if esperado:
        return ["false", "true"], [0.0, 1.0]
    return ["false", "true"], [1.0, 0.0]

def build_training_item(state, q, esperado, qid=None, soft=None):
    if qid:
        q = dict(q, _qid=qid)
    keys, target = gold_probs(q, esperado, soft)
    s = sum(target)
    target = [v / s for v in target] if s > 0 else [1.0 / len(target)] * len(target)
    label = target.index(max(target))
    k = len(render_options({"t": q["type"], "crit": q.get("criteria", {})}))
    seq, markers = build_sequence(tok, state, {"t": q["type"],
                                               "ins": q["instructions"],
                                               "crit": q.get("criteria", {})},
                                  cfg["max_len"], cfg["head_max_len"])
    if len(markers) != k:
        return None
    return {"ids": seq, "markers": markers, "qtype": QTYPES[q["type"]],
            "target": target, "label": label}

def itens_de(registros):
    itens = []
    for reg in registros:
        for qid, q in reg["questions"].items():
            if qid in reg["expected"]:
                it = build_training_item(reg["state"], q, reg["expected"][qid],
                                         qid=qid, soft=reg.get("soft"))
                if it:
                    itens.append(it)
    return itens

import os
# sessao nova do Colab tem /content/ VAZIO — baixa os 6 arquivos do repo
# publico lucianfialho/laya-finetune automaticamente. Arquivos ja
# arrastados pro painel sao respeitados (so baixa o que falta).
import urllib.request
BASE_RAW = "https://raw.githubusercontent.com/lucianfialho/laya-finetune/main"
ARQUIVOS = ["distill_treino.jsonl", "distill_er_treino.jsonl",
            "er_volume.jsonl", "classificacao_heldout.jsonl",
            "fila_610.jsonl", "baseline_typed-decisions.json"]
os.makedirs("/content/data", exist_ok=True)
for nome in ARQUIVOS:
    destino = f"/content/data/{nome}"
    painel = f"/content/{nome}"
    if not os.path.exists(destino) and not os.path.exists(painel):
        print(f"baixando {nome}...", end=" ", flush=True)
        urllib.request.urlretrieve(f"{BASE_RAW}/{nome}", destino)
        print("ok", flush=True)

# arquivos arrastados pro painel lateral podem cair em /content/ ou em
# qualquer subpasta (ex.: sample_data/) — varre tudo e centraliza em
# /content/data/, relatando o que achou e o que falta
import glob, shutil
NOMES_ESPERADOS = ['distill_treino.jsonl', 'distill_er_treino.jsonl',
                   'er_volume.jsonl', 'classificacao_heldout.jsonl',
                   'fila_610.jsonl', 'baseline_typed-decisions.json']
achados = {}
for arq in glob.glob('/content/**/*', recursive=True):
    base = os.path.basename(arq)
    if base in NOMES_ESPERADOS and os.path.isfile(arq) and base not in achados:
        achados[base] = arq
for nome, origem in achados.items():
    if os.path.dirname(origem) != '/content/data':
        shutil.move(origem, f'/content/data/{nome}')
print(f"encontrados: {sorted(achados)}")
faltando = [n for n in NOMES_ESPERADOS if n not in achados]
if faltando:
    raise FileNotFoundError(
        f"FALTAM arquivos e o download automatico nao resolveu: {faltando}\n"
        "Sem er_volume.jsonl o treino REPETE o erro do Exp B (vies de merge). "
        "Confira a conexao ou arraste os arquivos pro painel lateral.")

def tem(nome):
    return os.path.exists(f'/content/data/{nome}')

# Exp B (destilacao) quando os arquivos com soft labels foram upados;
# senao Exp A (one-hot) — mesmo preprocess, mesmos gates
arq_cls = 'distill_treino.jsonl' if tem('distill_treino.jsonl') \
          else 'classificacao_treino.jsonl'
arq_er = 'distill_er_treino.jsonl' if tem('distill_er_treino.jsonl') \
         else 'er_treino.jsonl'
treino_cls = carregar_jsonl(f'/content/data/{arq_cls}')
treino_er = carregar_jsonl(f'/content/data/{arq_er}')
# Exp B v2: volume balanceado de ER (2.550+ pares corroborados com soft
# label do Jev — corrige o vies de merge do Exp B, ver resultado_treino.json)
if tem('er_volume.jsonl'):
    treino_er = treino_er + carregar_jsonl('/content/data/er_volume.jsonl')
items = itens_de(treino_cls) + itens_de(treino_er)
n_soft = sum(1 for r in treino_cls + treino_er if r.get('soft'))
print(f"{len(items)} sequencias de treino ({arq_cls} + {arq_er}"
      f"{' + er_volume' if tem('er_volume.jsonl') else ''}; "
      f"{n_soft} registros com soft labels do Jev)")

# calibracao de temperatura: mesma receita do notebook oficial (seed fixa,
# 10% do treino fica de fora so pra fit de temperatura — DIFERENTE do
# held-out, que fica no repo e e o gate de qualidade)
order = list(range(len(items)))
random.Random(20260922).shuffle(order)
n_calib = min(400, len(items) // 10)
calib_items = [items[i] for i in sorted(order[:n_calib])]
train_items = [items[i] for i in sorted(order[n_calib:])]
torch.save({"train": train_items, "calib": calib_items,
            "model_dir": model_dir, "td_dir": td_dir},
           '/content/train_bundle.pt')
print(f"{len(train_items)} treino | {len(calib_items)} calibracao")

## 2. Treino (T4 unica)

In [ ]:
%%writefile /content/train_single.py
# Loop de treino adaptado do notebook oficial (2xT4 DDP -> T4 unica).
# Mesma matematica: RLCD com proper_reward (w_sph=0.75), sampling sigma
# 0.4->0.1, CE guidance 1.0, AdamW com LR separado encoder/head, cosine.
import os, sys, time, json, random
import torch
from safetensors.torch import load_file, save_file
from transformers import AutoTokenizer
from laya.common import build_model, proper_reward, QTYPES

def collate_train_batch(items, pad_id):
    n, L = len(items), max(len(it["ids"]) for it in items)
    kmax = max(len(it["markers"]) for it in items)
    ids = torch.full((n, L), pad_id, dtype=torch.long)
    att = torch.zeros((n, L), dtype=torch.long)
    mpos = torch.zeros((n, kmax), dtype=torch.long)
    mmask = torch.zeros((n, kmax), dtype=torch.bool)
    target = torch.zeros((n, kmax), dtype=torch.float32)
    for i, it in enumerate(items):
        ids[i, : len(it["ids"])] = torch.tensor(it["ids"])
        att[i, : len(it["ids"])] = 1
        k = len(it["markers"])
        mpos[i, :k] = torch.tensor(it["markers"])
        mmask[i, :k] = True
        target[i, : len(it["target"])] = torch.tensor(it["target"], dtype=torch.float32)
    return {"input_ids": ids, "attention_mask": att, "marker_pos": mpos,
            "marker_mask": mmask, "target": target,
            "qtype": torch.tensor([it["qtype"] for it in items]),
            "label": torch.tensor([it["label"] for it in items])}

def fit_one_temp(sel):
    if len(sel) < 10:
        return 1.0
    kmax = max(len(z) for z, _ in sel)
    Z = torch.full((len(sel), kmax), -1e4)
    T = torch.zeros((len(sel), kmax))
    for i, (z, t) in enumerate(sel):
        Z[i, :len(z)] = torch.tensor(z)
        T[i, :len(t)] = torch.tensor(t, dtype=torch.float32)
    log_t = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([log_t], lr=0.1, max_iter=100)
    def closure():
        opt.zero_grad()
        loss = -(T * torch.log_softmax(Z / log_t.exp(), -1)).sum(-1).mean()
        loss.backward()
        return loss
    opt.step(closure)
    return float(torch.clamp(log_t.exp(), 0.1, 10.0).item())

def main():
    bundle = torch.load('/content/train_bundle.pt', weights_only=False)
    train_items, calib_items = bundle["train"], bundle["calib"]
    model_dir = bundle["model_dir"]
    output_dir = "/content/laya_wiki_docs"

    with open(os.path.join(model_dir, "rl_agent_config.json")) as f:
        cfg = json.load(f)
    cfg["gradient_checkpointing"] = True
    cfg["max_tokens_per_batch"] = 4096
    cfg["max_len"] = 1024
    cfg["head_max_len"] = 256

    tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
    encoder_dir = os.path.join(model_dir, "typed-decisions", "encoder")
    if not os.path.isdir(encoder_dir):
        encoder_dir = os.path.join(model_dir, "encoder")
    model = build_model(cfg, encoder_dir=encoder_dir)
    model.load_state_dict(load_file(os.path.join(model_dir, "typed-decisions",
                                                 "model.safetensors")), strict=True)
    model.encoder.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={"use_reentrant": False})
    model.head_checkpointing = True
    device = torch.device("cuda")
    model.to(device)
    model.train()

    EPOCHS = 4
    MICRO_BATCH = 8
    GRAD_ACCUM = 2      # batch efetivo 16 (dataset pequeno)
    GROUP_SIZE = 4
    LR_ENCODER = 2.5e-5
    LR_HEAD = 1.0e-4
    SIGMA_START, SIGMA_END = 0.4, 0.1

    enc_params = [p for n, p in model.named_parameters() if "encoder." in n]
    head_params = [p for n, p in model.named_parameters() if "encoder." not in n]
    optimizer = torch.optim.AdamW([
        {"params": enc_params, "lr": LR_ENCODER},
        {"params": head_params, "lr": LR_HEAD}], weight_decay=0.01)
    total_updates = (len(train_items) // (MICRO_BATCH * GRAD_ACCUM)) * EPOCHS
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max(1, total_updates), eta_min=1e-6)
    scaler = torch.amp.GradScaler("cuda", enabled=True)

    print(f"T4 unica: {len(train_items)} itens ({len(calib_items)} calibracao) | "
          f"{EPOCHS} epocas")
    t0 = time.time()
    for epoch in range(EPOCHS):
        random.seed(42 + epoch)
        random.shuffle(train_items)
        epoch_loss, n_batches = 0.0, 0
        optimizer.zero_grad(set_to_none=True)
        accum_step = 0
        sigma = SIGMA_START + (SIGMA_END - SIGMA_START) * epoch / max(1, EPOCHS - 1)

        for b_idx in range(0, len(train_items), MICRO_BATCH):
            chunk = train_items[b_idx:b_idx + MICRO_BATCH]
            if not chunk:
                continue
            batch = collate_train_batch(chunk, tok.pad_token_id)
            with torch.autocast("cuda", dtype=torch.float16):
                logits, act = model(batch["input_ids"].to(device),
                                    batch["attention_mask"].to(device),
                                    batch["marker_pos"].to(device),
                                    batch["marker_mask"].to(device),
                                    batch["qtype"].to(device))
            logits = logits.float()
            mask = batch["marker_mask"].to(device)
            k = mask.sum(-1, keepdim=True).float()
            target = batch["target"].to(device)

            eps = torch.randn((GROUP_SIZE,) + logits.shape, device=device) * sigma * mask
            eps = (eps - eps.sum(-1, keepdim=True) / k) * mask
            z = logits.detach().unsqueeze(0) + eps
            q = torch.softmax(z.masked_fill(~mask, -1e4), -1)
            with torch.no_grad():
                r = proper_reward(q, target.unsqueeze(0), batch["qtype"].to(device),
                                  mask, w_sph=0.75, w_rps=1.0)
                adv = r - r.mean(0, keepdim=True)
                adv = adv / (adv.std() + 1e-6)
            logp = -(((z - logits.unsqueeze(0)) ** 2) * mask).sum(-1) / (2 * sigma ** 2)
            loss_rl = -(adv * logp).mean()
            loss_ce = -(target * torch.log_softmax(
                logits.masked_fill(~mask, -1e4), -1)).sum(-1).mean()
            loss = (loss_rl + 1.0 * loss_ce) / GRAD_ACCUM + 0.0 * act.sum()
            scaler.scale(loss).backward()
            accum_step += 1
            if accum_step % GRAD_ACCUM == 0 or (b_idx + MICRO_BATCH) >= len(train_items):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
            epoch_loss += loss.item() * GRAD_ACCUM
            n_batches += 1
            if (n_batches % 20) == 0:
                print(f"  Epoca {epoch+1}/{EPOCHS} | Lote {n_batches} | "
                      f"Loss: {loss.item()*GRAD_ACCUM:.4f} | "
                      f"Reward: {r.mean().item():.3f}", flush=True)
        print(f"=== Epoca {epoch+1}/{EPOCHS} em {time.time()-t0:.0f}s | "
              f"Loss media: {epoch_loss/max(1,n_batches):.4f} ===", flush=True)
        ckpt = os.path.join(output_dir, "checkpoint_latest")
        os.makedirs(ckpt, exist_ok=True)
        sd = {kk: v.half().contiguous().cpu() for kk, v in model.state_dict().items()}
        save_file(sd, os.path.join(ckpt, "model.safetensors"))
        model.encoder.config.save_pretrained(os.path.join(ckpt, "encoder"))
        tok.save_pretrained(os.path.join(ckpt, "tokenizer"))
        print(f"  checkpoint rodando salvo (epoca {epoch+1})", flush=True)

    print("Calibracao de temperatura (slice separada, nunca treinada)...")
    del optimizer, scaler, scheduler
    torch.cuda.empty_cache()
    model.eval()
    calib_preds = []
    with torch.no_grad():
        for c_idx in range(0, len(calib_items), 16):
            c_chunk = calib_items[c_idx:c_idx + 16]
            cb = collate_train_batch(c_chunk, tok.pad_token_id)
            with torch.autocast("cuda", dtype=torch.float16):
                l_sub, _ = model(cb["input_ids"].to(device),
                                 cb["attention_mask"].to(device),
                                 cb["marker_pos"].to(device),
                                 cb["marker_mask"].to(device),
                                 cb["qtype"].to(device))
            l_np = l_sub.float().cpu().numpy()
            for ri, it in enumerate(c_chunk):
                kk = len(it["markers"])
                calib_preds.append((it["qtype"], l_np[ri, :kk], it["target"]))
    fitted = [1.2, 1.2, 1.2]
    for qt in range(3):
        sel = [(z, t) for q_type, z, t in calib_preds if q_type == qt]
        if sel:
            fitted[qt] = fit_one_temp(sel)
    print("Temperaturas (choice, score, noul):", [round(t, 3) for t in fitted])

    os.makedirs(output_dir, exist_ok=True)
    sd = {kk: v.half().contiguous().cpu() for kk, v in model.state_dict().items()}
    save_file(sd, os.path.join(output_dir, "model.safetensors"))
    model.encoder.config.save_pretrained(os.path.join(output_dir, "encoder"))
    tok.save_pretrained(os.path.join(output_dir, "tokenizer"))
    cfg["fine_tuned"] = True
    cfg["model_name"] = "laya-typed-decisions"
    cfg["temperature"] = fitted
    cfg.pop("temperature_by_options", None)
    with open(os.path.join(output_dir, "rl_agent_config.json"), "w") as f:
        json.dump(cfg, f, indent=2)
    print(f"Modelo salvo em {output_dir}")

if __name__ == "__main__":
    main()

In [ ]:
import subprocess, torch, time
assert torch.cuda.is_available(), \
    "Runtime sem GPU! Troque para T4 (Runtime > Change runtime type)"
print("treino rodando — saida ao vivo abaixo (epocas ~5-10 min cada):\n")
t0 = time.time()
proc = subprocess.Popen(
    ["python", "/content/train_single.py"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for linha in proc.stdout:          # stream ao vivo, linha a linha
    print(linha, end="", flush=True)
proc.wait()
print(f"\nprocesso terminou em {time.time()-t0:.0f}s "
      f"(codigo {proc.returncode})")
assert proc.returncode == 0, "treino falhou — veja a saida acima"


## 2. Gates de avaliação (ver `SUCESSO.md` no repo — escritos ANTES do treino)

Gate 1 (primário, held-out gold adversarial): bater `ga4_current` ≥ 0.60
(baseline 0.283) sem regredir `product_line` além de 5%.
Gate 2 (operacional, fila_610): cobertura auto-aceite (margem ≥ 0.15 em
product_line E em ga4_current) maior que a do base, com accuracy ≥ 0.95 no
subconjunto auto-aceito. É o motivo real do fine-tune.
Gate 3 (guarda-redes, ER): F1 same-class nos 45 pares do benchmark não
regrediu.

In [ ]:
import json, laya

heldout = [json.loads(l) for l in open('/content/data/classificacao_heldout.jsonl')
           if l.strip()]
fila = [json.loads(l) for l in open('/content/data/fila_610.jsonl') if l.strip()]
benchmark = [r for r in (json.loads(l) for l in open('/content/data/er_treino.jsonl'))
             if 'benchmark' in r.get('tags', [])]
baseline = json.load(open('/content/data/baseline_typed-decisions.json'))["metricas"]

agent = laya.load('/content/laya_wiki_docs')

def respostas_de(agent, reg):
    return agent.predict({"body": reg["state"]}, reg["questions"])["answers"]

def margin_de(ans, q):
    if q["type"] == "choice":
        probs = sorted((float(v) for v in ans["probabilities"].values()),
                       reverse=True)
        return probs[0] - probs[1]
    return 2 * abs(float(ans["noul"]) - 0.5)  # margem 0-1 comparavel ao choice

# ---- gate 1: held-out ----
acertos, totais = {}, {}
for reg in heldout:
    resp = respostas_de(agent, reg)
    for qid, esperado in reg["expected"].items():
        ans = resp[qid]
        obtido = ans["choice"] if reg["questions"][qid]["type"] == "choice" \
                 else float(ans["noul"]) >= 0.5
        totais[qid] = totais.get(qid, 0) + 1
        acertos[qid] = acertos.get(qid, 0) + (obtido == esperado)

print(f"{'GATE 1 held-out':15s} {'treinado':>10s} {'baseline':>10s}")
gate1 = True
for qid in sorted(totais):
    acc = acertos[qid] / totais[qid]
    base = baseline.get(f"{qid}[gold]", {}).get("accuracy")
    marca = ""
    if base is not None:
        if qid == "ga4_current" and acc < 0.60:
            marca = "  <- FALHOU (gate 0.60)"; gate1 = False
        elif qid != "ga4_current" and acc < base - 0.05:
            marca = "  <- REGRESSAO"; gate1 = False
    print(f"{qid:15s} {acc:10.3f} {base if base is not None else 'n/a':>10} {marca}")

# ---- gate 2: cobertura nos 610, accuracy do auto-aceite contra GOLD ----
# (v2: silver nao e verdade — medir accuracy contra os rotulos que o
# professor Jev discorda puniria o modelo por estar certo. Gold puro
# = held-out, que nunca entrou no treino.)
CORTE = 0.15
decididos = 0
for reg in fila:
    resp = respostas_de(agent, reg)
    if all(margin_de(resp[qid], reg["questions"][qid]) >= CORTE
           for qid in ("product_line", "ga4_current")):
        decididos += 1
cobertura = decididos / len(fila)

n_auto_gold, corretos_auto_gold = 0, 0
for reg in heldout:
    resp = respostas_de(agent, reg)
    for qid in ("product_line", "ga4_current"):
        q = reg["questions"][qid]
        if margin_de(resp[qid], q) >= CORTE:
            n_auto_gold += 1
            obtido = (resp[qid]["choice"] if q["type"] == "choice"
                      else float(resp[qid]["noul"]) >= 0.5)
            corretos_auto_gold += (obtido == reg["expected"][qid])
acc_auto = corretos_auto_gold / max(1, n_auto_gold)
print(f"\nGATE 2 fila: cobertura auto-aceite {cobertura:.1%} "
      f"(decididos {decididos}/{len(fila)}) | "
      f"accuracy auto-aceite (gold, {n_auto_gold} eixos): {acc_auto:.3f}")
print("  (comparar com a cobertura do base — rodar a mesma celula apontando")
print("   laya.load com o snapshot original pra ter o par base/treinado)")

# ---- gate 3: ER benchmark ----
tp = fp = fn = 0
for reg in benchmark:
    ans = respostas_de(agent, reg)["mesmo"]
    pred = ans["choice"]
    gold = reg["expected"]["mesmo"]
    tp += pred == "same" and gold == "same"
    fp += pred == "same" and gold != "same"
    fn += pred != "same" and gold == "same"
p = tp / (tp + fp) if tp + fp else 0
r = tp / (tp + fn) if tp + fn else 0
f1 = 2 * p * r / (p + r) if p + r else 0
print(f"\nGATE 3 ER benchmark: F1 same-class {f1:.3f} (P={p:.3f} R={r:.3f})")

print()
print("GATE 1:", "PASSOU" if gate1 else "FALHOU")
print("(decisao final ADOTA/NÃO ADOTA: registrar em laya-treino/resultado_treino.json")

## 3. (Opcional) Push pro Hugging Face

Crie um token em hf.co/settings/tokens (escopo `write`) e um model repo
(p.ex. `SEU_USUARIO/laya-wiki-docs`). O checkpoint vira um subfolder —
no pipeline, trocar `subfolder="typed-decisions"` por ele.

In [ ]:
import os
from huggingface_hub import HfApi

if "HUGGINGFACE_TOKEN" not in os.environ:
    from google.colab import userdata
    try:
        os.environ["HUGGINGFACE_TOKEN"] = userdata.get("HUGGINGFACE_TOKEN")
    except Exception:
        os.environ["HUGGINGFACE_TOKEN"] = input("HF token (escopo write): ")
REPO = input("Repo (ex.: SEU_USUARIO/laya-wiki-docs): ")
api = HfApi(token=os.environ["HUGGINGFACE_TOKEN"])
api.create_repo(REPO, exist_ok=True)
api.upload_folder(folder_path="/content/laya_wiki_docs",
                  repo_id=REPO, path_in_repo="typed-decisions")
print(f"Pushado em https://huggingface.co/{REPO}/tree/main/typed-decisions")
print("Adocao no pipeline: Agent('convaiinnovations/laya', subfolder=...)")
print(f"  -> Agent('{REPO}', subfolder='typed-decisions')")